# HVAE Synthetic Genomic Model
Clean, runnable version for Colab / Kaggle.

Includes:
- Synthetic genomic dataset with motifs
- One-hot encoding
- Hierarchical VAE (HVAE)
- Training loop
- Latent visualisation (PCA)


In [ ]:
# Install dependencies
!pip install -q numpy torch scikit-learn matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Synthetic Genomic Data

In [ ]:
def generate_sequence(length=100):
    bases = ['A','C','G','T']
    seq = [np.random.choice(bases) for _ in range(length)]

    # inject motif
    if np.random.rand() > 0.5:
        pos = np.random.randint(0, length-6)
        seq[pos:pos+6] = list("ATGCGT")

    return "".join(seq)

def create_dataset(n=1000, length=100):
    return [generate_sequence(length) for _ in range(n)]

data = create_dataset()
len(data)

## Encoding

In [ ]:
mapping = {'A':0,'C':1,'G':2,'T':3}

def encode(seq):
    arr = np.zeros((len(seq), 4))
    for i, base in enumerate(seq):
        arr[i, mapping[base]] = 1
    return arr.flatten()

X = np.array([encode(s) for s in data])
X.shape

## HVAE Model

In [ ]:
class HVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, z1_dim=32, z2_dim=16):
        super().__init__()

        self.enc1 = nn.Linear(input_dim, hidden_dim)
        self.enc_z1_mu = nn.Linear(hidden_dim, z1_dim)
        self.enc_z1_logvar = nn.Linear(hidden_dim, z1_dim)

        self.enc2 = nn.Linear(z1_dim, hidden_dim)
        self.enc_z2_mu = nn.Linear(hidden_dim, z2_dim)
        self.enc_z2_logvar = nn.Linear(hidden_dim, z2_dim)

        self.dec2 = nn.Linear(z2_dim, hidden_dim)
        self.dec_z1 = nn.Linear(hidden_dim, z1_dim)

        self.dec1 = nn.Linear(z1_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, input_dim)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h1 = F.relu(self.enc1(x))
        z1_mu, z1_logvar = self.enc_z1_mu(h1), self.enc_z1_logvar(h1)
        z1 = self.reparam(z1_mu, z1_logvar)

        h2 = F.relu(self.enc2(z1))
        z2_mu, z2_logvar = self.enc_z2_mu(h2), self.enc_z2_logvar(h2)
        z2 = self.reparam(z2_mu, z2_logvar)

        d2 = F.relu(self.dec2(z2))
        z1_recon = self.dec_z1(d2)

        d1 = F.relu(self.dec1(z1_recon))
        x_recon = torch.sigmoid(self.out(d1))

        return x_recon, z1_mu, z1_logvar, z2_mu, z2_logvar


## Loss Function

In [ ]:
def loss_fn(x_recon, x, z1_mu, z1_logvar, z2_mu, z2_logvar):
    recon = F.binary_cross_entropy(x_recon, x, reduction='sum')

    kl1 = -0.5 * torch.sum(1 + z1_logvar - z1_mu.pow(2) - z1_logvar.exp())
    kl2 = -0.5 * torch.sum(1 + z2_logvar - z2_mu.pow(2) - z2_logvar.exp())

    return recon + kl1 + kl2


## Training

In [ ]:
X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

model = HVAE(input_dim=X.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()

    optimizer.zero_grad()
    x_recon, z1_mu, z1_logvar, z2_mu, z2_logvar = model(X_tensor)

    loss = loss_fn(x_recon, X_tensor, z1_mu, z1_logvar, z2_mu, z2_logvar)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.2f}")


## Latent Space Visualisation

In [ ]:
model.eval()
with torch.no_grad():
    h1 = F.relu(model.enc1(X_tensor))
    z1_mu = model.enc_z1_mu(h1).cpu().numpy()

pca = PCA(n_components=2)
z_pca = pca.fit_transform(z1_mu)

plt.scatter(z_pca[:,0], z_pca[:,1], s=5)
plt.title("Latent Space PCA")
plt.show()
